# Part 3: Milvus Application Practice - RAG, Agent, and Image Search Use Cases

Welcome to Part 3 of the Milvus Workshop! In this section, we will apply the Milvus fundamentals learned previously to some practical use cases, including image search, and potentially expanding to RAG (Retrieval Augmented Generation) and Agent applications in the future.

## 3.1 Milvus in Image Search Applications [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/richzw/milvus-workshop/blob/main/ch3/ch3_1_en.ipynb)

Image search is a classic application scenario for vector databases. Whether it's "image-to-image search" (finding images similar to a given image) or "text-to-image search" (finding matching images based on text descriptions), the core principle is to convert images (or text) into high-dimensional vectors (embeddings), and then perform similarity searches in the vector space.

### Workflow: Image Loading -> Image Vectorization (Embedding) -> Storage -> Retrieval

A typical image search system usually includes the following steps:

1.  **Image Loading and Preprocessing**:
    *   Load images from disk, URLs, or databases.
    *   Perform necessary preprocessing on images, such as resizing, normalization, etc., to match the input requirements of pre-trained models.

2.  **Image Vectorization (Embedding)**:
    *   Use pre-trained deep learning models (such as ResNet, VGG, EfficientNet, CLIP, ViT, etc.) to extract feature vectors from images.
    *   These models can convert the visual content of images into fixed-dimensional numerical vectors, where similar images will be closer together in the vector space.
    *   For "text-to-image search", multimodal models (such as CLIP) that can process both text and images are needed, converting text descriptions into the same semantic space as image vectors.

3.  **Storage (Ingestion to Milvus)**:
    *   Store the unique identifier of each image (such as file path, ID) and its corresponding feature vector in a Milvus Collection.
    *   Create appropriate indexes for vector fields to accelerate subsequent retrieval processes.

4.  **Retrieval (Search/Query)**:
    *   **Image-to-Image Search**:
        1.  Perform the same loading, preprocessing, and vectorization operations on the query input image as in steps 1 and 2 to obtain the query vector.
        2.  Use the query vector to perform vector similarity search in Milvus, finding the Top-K most similar image vectors.
        3.  Return the identifiers of these similar images.
    *   **Text-to-Image Search (requires multimodal model)**:
        1.  Use the text encoder of a multimodal model to convert the input text description into a query vector.
        2.  Use the query vector to perform vector similarity search in Milvus (where image vectors are stored).
        3.  Return the identifiers of images most relevant to the text description.

### Milvus's Role in Image Search

- **Storing Image Feature Vectors**: Milvus efficiently stores and manages millions or even billions of image feature vectors generated by deep learning models.
- **Implementing Efficient Similarity Search**: Utilizing its built-in multiple vector indexes and ANNS algorithms, Milvus can quickly retrieve results most similar to query vectors (from images or text) from massive image vectors.
- **Scalability and Reliability**: Milvus's distributed architecture enables it to handle large-scale image datasets and provide high availability and data persistence.

### Case Demo/Code Explanation: Building a Simple Image Search Demo

We will build a very basic "image-to-image search" demonstration.

**Steps Overview:**
1.  **Environment Setup**: Install necessary libraries.
2.  **Prepare Image Dataset**: Use a small set of sample images.
3.  **Select Pre-trained Model**: We will use a pre-trained ResNet model provided by `torchvision` to extract features, with simple dimensionality reduction (optional). For a more modern demonstration, using CLIP would be better, but its setup and dependencies are slightly more complex.
4.  **Create Milvus Collection**: To store image paths and vectors.
5.  **Image Vectorization and Insertion into Milvus**.
6.  **Implement Search Functionality**.

#### 1. Environment Setup and Library Imports

**Please ensure you have installed the following libraries:**
```bash
pip install timm==1.0.15 torch==2.7.0 numpy scikit-learn==1.5.1 pillow==10.4.0 pymilvus
```

In [1]:
import os
import random
import shutil
import torch
from PIL import Image
import timm
from sklearn.preprocessing import normalize
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# Milvus Client
from pymilvus import MilvusClient, DataType, FieldSchema, CollectionSchema

# For displaying images (in Jupyter Notebook)
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Pillow version: {Image.__version__}")

ModuleNotFoundError: No module named 'timm'

#### 2. Prepare Image Dataset

We need some images. For demonstration purposes, we can download a few sample images from the web, or use local images.
Here we create a simple function to download some sample images (if the runtime environment allows network access).

In [ ]:
# --- Image Dataset Preparation ---
IMAGE_DATA_DIR = "milvus_image_search_data"
SAMPLE_IMAGE_URLS = [
    "https://images.unsplash.com/photo-1583511655826-05700d52f4d9?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Dog
    "https://images.unsplash.com/photo-1548199973-03cce0bbc87b?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Dogs playing
    "https://images.unsplash.com/photo-1503023345310-bd7c1de61c7d?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Person
    "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Cat
    "https://images.unsplash.com/photo-1573865526739-10659fec78a5?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Another Cat
    "https://images.unsplash.com/photo-1481349518771-20055b2a7b24?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Banana
    "https://images.unsplash.com/photo-1528825871115-3581a5387919?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # More Bananas
    "https://images.unsplash.com/photo-1502672260266-1c1ef2d93688?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Cityscape
    "https://images.unsplash.com/photo-1506260408121-e353d10b87c7?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max", # Landscape
    "https://images.unsplash.com/photo-1533743983669-94fa5c4338ec?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=400&fit=max"  # Car
]

# Clean and create image directory
if os.path.exists(IMAGE_DATA_DIR):
    shutil.rmtree(IMAGE_DATA_DIR)
os.makedirs(IMAGE_DATA_DIR, exist_ok=True)

import requests
def download_image(url, folder, filename):
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        filepath = os.path.join(folder, filename)
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        # print(f"Downloaded {filename}")
        return filepath
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None

image_paths = []
print("Preparing to download sample images...")
for i, url in enumerate(SAMPLE_IMAGE_URLS):
    filename = f"sample_image_{i+1}.jpg"
    path = download_image(url, IMAGE_DATA_DIR, filename)
    if path:
        image_paths.append(path)

if not image_paths:
    print("Failed to download any sample images. Please ensure network connectivity or manually add images to the '" + IMAGE_DATA_DIR + "' directory.")
    # Here you can guide users to manually add images or stop
    # raise RuntimeError("Unable to obtain sample images, demo cannot continue.")
else:
    print(f"\nSuccessfully downloaded/prepared {len(image_paths)} sample images to '{IMAGE_DATA_DIR}' directory.")
    # Display a few downloaded images
    fig, axes = plt.subplots(1, min(5, len(image_paths)), figsize=(15, 3))
    if len(image_paths) == 1 and min(5, len(image_paths)) == 1: # Handle the case of only one image
        axes = [axes]
    for i, img_path in enumerate(image_paths[:5]):
        try:
            img = Image.open(img_path).convert("RGB")
            axes[i].imshow(img)
            axes[i].set_title(os.path.basename(img_path))
            axes[i].axis('off')
        except Exception as e:
            print(f"Cannot display image {img_path}: {e}")
    plt.show()

#### 3. Select Pre-trained Model and Define Feature Extraction Function

We use the ResNet-34 model to extract embedding information from images.

In [ ]:
import torch
from PIL import Image
import timm
from sklearn.preprocessing import normalize
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform


class FeatureExtractor:
    def __init__(self, modelname):
        # Load the pre-trained model
        self.model = timm.create_model(
            modelname, pretrained=True, num_classes=0, global_pool="avg"
        )
        self.model.eval()

        # Get the input size required by the model
        self.input_size = self.model.default_cfg["input_size"]

        config = resolve_data_config({}, model=modelname)
        # Get the preprocessing function provided by TIMM for the model
        self.preprocess = create_transform(**config)

    def __call__(self, imagepath):
        # Preprocess the input image
        input_image = Image.open(imagepath).convert("RGB")  # Convert to RGB if needed
        input_image = self.preprocess(input_image)

        # Convert the image to a PyTorch tensor and add a batch dimension
        input_tensor = input_image.unsqueeze(0)

        # Perform inference
        with torch.no_grad():
            output = self.model(input_tensor)

        # Extract the feature vector
        feature_vector = output.squeeze().numpy()

        return normalize(feature_vector.reshape(1, -1), norm="l2").flatten()

#### 4. Create Milvus Collection

We need a Collection to store image paths (as primary keys) and extracted feature vectors.

In [ ]:
MILVUS_HOST_IMG = "localhost"
MILVUS_PORT_IMG = "19530"
MILVUS_URI_IMG = f"http://{MILVUS_HOST_IMG}:{MILVUS_PORT_IMG}"

IMG_COLLECTION_NAME = "milvus_image_demo_collection"
IMG_ID_FIELD = "image_id" # Store image path or unique ID
IMG_VECTOR_FIELD = "image_vector"
VECTOR_DIMENSION = 512

# Connect to Milvus
try:
    img_milvus_client = MilvusClient(uri=MILVUS_URI_IMG)
    print(f"Successfully connected to Milvus service: {MILVUS_URI_IMG}")
except Exception as e:
    print(f"Failed to connect to Milvus service: {e}")
    raise

# Check and drop old Collection (for demo re-entrancy)
if img_milvus_client.has_collection(collection_name=IMG_COLLECTION_NAME):
    print(f"Found existing Collection '{IMG_COLLECTION_NAME}', dropping it.")
    img_milvus_client.drop_collection(collection_name=IMG_COLLECTION_NAME)

# Define Schema
# Primary key: Store image path or unique ID, VARCHAR type
id_field = FieldSchema(name=IMG_ID_FIELD, dtype=DataType.VARCHAR, is_primary=True, max_length=1024, auto_id=False)
# Vector field: Store image feature vectors
vector_field = FieldSchema(name=IMG_VECTOR_FIELD, dtype=DataType.FLOAT_VECTOR, dim=VECTOR_DIMENSION)

schema = CollectionSchema(
    fields=[id_field, vector_field],
    description="Collection for image similarity search demo",
    enable_dynamic_field=False # Usually set to False for fixed Schema
)

# Create Collection
try:
    img_milvus_client.create_collection(
        collection_name=IMG_COLLECTION_NAME,
        schema=schema,
        consistency_level="Strong" # Ensure immediate query after insertion
    )
    print(f"Collection '{IMG_COLLECTION_NAME}' created successfully.")
    print(f"Schema: ID({IMG_ID_FIELD}, VARCHAR), Vector({IMG_VECTOR_FIELD}, FLOAT_VECTOR, dim={VECTOR_DIMENSION})")
except Exception as e:
    print(f"Failed to create Collection '{IMG_COLLECTION_NAME}': {e}")
    raise

#### 5. Image Vectorization and Insertion into Milvus

In [ ]:
import os

extractor = FeatureExtractor("resnet34")

root = "./"+IMAGE_DATA_DIR

print(f"\nPreparing to insert image feature vectors into Milvus...")
for dirpath, foldername, filenames in os.walk(root):
    for filename in filenames:
        if filename.endswith(".jpg"):
            filepath = dirpath + "/" + filename
            image_embedding = extractor(filepath)
            img_milvus_client.insert(
                IMG_COLLECTION_NAME,
                {IMG_VECTOR_FIELD: image_embedding, IMG_ID_FIELD: filepath},
            )

print("Flushing collection...")
img_milvus_client.flush(collection_name=IMG_COLLECTION_NAME)
print("Flush completed.")

# Check entity count
stats_after_insert = img_milvus_client.get_collection_stats(collection_name=IMG_COLLECTION_NAME)
print(f"Collection '{IMG_COLLECTION_NAME}' current entity count: {stats_after_insert.get('row_count')}")

#### 5.1 (Important) Create Index for Vector Field

For efficient searching, we need to create an index for `IMG_VECTOR_FIELD`.

In [ ]:
# --- Create Index ---
IDX_NAME = "idx_img_embedding_hnsw"

if img_milvus_client.has_collection(IMG_COLLECTION_NAME) and int(img_milvus_client.get_collection_stats(IMG_COLLECTION_NAME).get('row_count',0)) > 0 :
    print(f"\nCreating index for field '{IMG_VECTOR_FIELD}' in Collection '{IMG_COLLECTION_NAME}'...")
    
    # Check if index already exists, if so, drop it first
    existing_indexes_img = img_milvus_client.list_indexes(collection_name=IMG_COLLECTION_NAME)
    if any(idx_info == IDX_NAME for idx_info in existing_indexes_img):
        print(f"Index already exists on field '{IMG_VECTOR_FIELD}', dropping old index.")
        try:
            img_milvus_client.drop_index(collection_name=IMG_COLLECTION_NAME, index_name=IDX_NAME)
            print("Old index dropped.")
        except Exception as e_drop_idx:
            print(f"Failed to drop old index: {e_drop_idx}")

    # HNSW index parameters
    img_index_params = MilvusClient.prepare_index_params()

    img_index_params.add_index(
        field_name=IMG_VECTOR_FIELD,
        metric_type="L2",
        index_type="HNSW",
        index_name=IDX_NAME,
        params={
            "M": 8,              # Maximum number of connections per node (smaller value, faster build)
            "efConstruction": 100 # Search scope during graph construction (smaller value, faster build)
        }
    )
    try:
        img_milvus_client.create_index(
            collection_name=IMG_COLLECTION_NAME,
            index_params=img_index_params
        )
        print(f"Index creation request sent. Parameters: {img_index_params}")
        
        # Wait for index build to complete (for small datasets, this is usually quick)
        print("Waiting for index build to complete...")

    except Exception as e:
        print(f"Failed to create index: {e}")
else:
    print(f"\nCollection '{IMG_COLLECTION_NAME}' does not exist or is empty, skipping index creation.")

#### 6. Implement Search Functionality (Image-to-Image Search)

In [ ]:
import warnings

# Suppress the specific UserWarning from pylabtools
warnings.filterwarnings("ignore", category=UserWarning, module="pylabtools", lineno=152)

def search_similar_images(query_image_path, top_k=3):
    if not img_milvus_client.has_collection(IMG_COLLECTION_NAME):
        print(f"Error: Collection '{IMG_COLLECTION_NAME}' does not exist.")
        return

    print(f"\nSearching for images similar to '{os.path.basename(query_image_path)}' (Top {top_k})...")

    # 1. Ensure Collection is loaded
    try:
        img_milvus_client.load_collection(collection_name=IMG_COLLECTION_NAME)
        # print(f"Collection '{IMG_COLLECTION_NAME}' loaded.")
    except Exception as e:
        print(f"Failed to load Collection: {e}")
        return

    # 2. Extract feature vector of query image
    query_vectors_for_search = [extractor(query_image_path)] # search API requires list of lists

    # 3. Define search parameters
    search_params_img = {
        "metric_type": "L2", # Consistent with index creation
        "params": {"ef": 32} # HNSW search parameter, ef >= top_k
    }

    # 4. Execute search
    try:
        results = img_milvus_client.search(
            collection_name=IMG_COLLECTION_NAME,
            data=query_vectors_for_search,
            anns_field=IMG_VECTOR_FIELD,
            limit=top_k,
            search_params=search_params_img,
            output_fields=[IMG_ID_FIELD] # We need image path/ID
        )
        
        # Display query image
        plt.figure(figsize=(4,4))
        query_img_display = Image.open(query_image_path).convert("RGB")
        plt.imshow(query_img_display)
        plt.title(f"Query Image: {os.path.basename(query_image_path)}")
        plt.axis('off')
        plt.show()

        print("\nSearch results:")
        if not results or not results[0]:
            print("  No similar images found.")
            return

        # Display result images
        num_results = len(results[0])
        fig, axes = plt.subplots(1, num_results, figsize=(num_results * 4, 4))
        if num_results == 1: axes = [axes] # Handle single result case

        for i, hit in enumerate(results[0]):
            result_id = hit.get(IMG_ID_FIELD) # This is the stored image identifier
            distance = hit.get('distance')
            
            result_image_path = result_id
            
            print(f"  - Result #{i+1}: ID='{result_id}', Distance={distance:.4f}, Path='{result_image_path}'")
            
            if os.path.exists(result_image_path):
                try:
                    img_display = Image.open(result_image_path).convert("RGB")
                    axes[i].imshow(img_display)
                    axes[i].set_title(f"Similar Image #{i+1}\nID: {os.path.basename(result_id)}\nDist: {distance:.2f}")
                    axes[i].axis('off')
                except Exception as e_disp:
                    print(f"    Cannot display image {result_image_path}: {e_disp}")
                    axes[i].text(0.5, 0.5, 'Cannot load image', ha='center', va='center')
                    axes[i].axis('off')
            else:
                print(f"    Warning: Result image path '{result_image_path}' does not exist.")
                axes[i].text(0.5, 0.5, f'Image not found:\n{os.path.basename(result_id)}', ha='center', va='center', fontsize=8)
                axes[i].axis('off')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error during search: {e}")

# Randomly select an image from the dataset as query image
if image_paths:
    random_query_image_path = random.choice(image_paths)
    search_similar_images(random_query_image_path, top_k=2) # Search Top4, the first one is usually itself
else:
    print("No images available for querying.")

### Hands-on Exercise 1: Principles of Cross-Modal Search (CLIP), and Selection of Different Image Embedding Models

- **Cross-Modal Search**:
    - **Principle**: Refers to querying in one modality (such as text) and retrieving results in another modality (such as images), or vice versa.
    - **CLIP (Contrastive Language-Image Pre-training)**: A representative model for implementing cross-modal search. CLIP performs contrastive learning on a large number of (image, text description) pairs, making images and their corresponding text descriptions have similar vector representations in the same shared vector space.
        - Image Encoder (such as ViT or ResNet) converts images into vectors.
        - Text Encoder (such as Transformer) converts text into vectors.
        - The training objective is to maximize the cosine similarity between vectors of matching (image, text) pairs while minimizing the similarity of non-matching pairs.
    - **Applications**:
        - **Text-to-Image Search**: Encode text descriptions into vectors, then search in Milvus which stores image vectors.
        - **Image-to-Text Search**: Encode images into vectors, then search in Milvus which stores text vectors.
        - **Zero-shot Image Classification**: Encode images, then compare them with text encodings of various category names, selecting the category with the highest similarity.

- **Selection of Different Image Embedding Models**:
    - **ResNet, VGG, EfficientNet, etc. (Supervised Classification Models)**:
        - Usually pre-trained on large-scale image classification datasets such as ImageNet.
        - Extracted features mainly focus on classification discriminability of images.
        - Good for general visual similarity search (e.g., similar objects, scenes, styles).
        - Usually requires removing the final classification layer, using the output of previous convolutional or pooling layers as features.
    - **SimCLR, MoCo, BYOL, etc. (Self-supervised Learning Models)**:
        - Trained by designing various pretext tasks (such as contrastive learning instance discrimination) on unlabeled data.
        - Learned features usually have better generalization ability and stronger adaptability to specific downstream tasks.
        - May perform better for certain specific types of similarity (such as fine-grained textures, instance-level recognition).
    - **CLIP, ALIGN, ViLT, etc. (Multimodal Models)**:
        - As mentioned above, trained through image-text pairs, learned features capture both visual and semantic information.
        - **Most suitable for cross-modal tasks** (text-to-image, image-to-text search).
        - For pure image-to-image search, their visual feature discriminability may be slightly inferior to supervised or self-supervised models specifically optimized for visual tasks, but they generally perform well and have the advantage of semantic understanding.
    - **Selection Criteria**:
        - **Task Type**: Is it image-to-image search or text-to-image search?
        - **Data Characteristics**: Domain and content type of images.
        - **Performance Requirements**: Search accuracy, speed, model size, inference latency.
        - **Available Resources**: Whether GPU is available, cost of obtaining and deploying pre-trained models.
        - **Experimentation**: Usually requires experimental evaluation of different models' effectiveness based on specific application scenarios.

Ref: https://milvus.io/docs/text_image_search.md